In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. Load Bronze Addresses
# ============================================================

bronze_addresses_df = spark.table(
    "customer360_dev.bronze.addresses"
)


# ============================================================
# 2. Find Latest Incremental Run
# ============================================================

latest_run_row = (
    bronze_addresses_df
    .filter(
        F.col("_load_type") == "INCREMENTAL"
    )
    .filter(
        F.col("_pipeline_run_id").isNotNull()
    )
    .groupBy(
        "_pipeline_run_id"
    )
    .agg(
        F.max("_ingestion_timestamp").alias(
            "latest_ingestion_timestamp"
        )
    )
    .orderBy(
        F.col("latest_ingestion_timestamp").desc()
    )
    .first()
)

run_id = (
    latest_run_row["_pipeline_run_id"]
    if latest_run_row
    else None
)

print("Processing Address Run ID:", run_id)


# ============================================================
# 3. Read Only Current Address Incremental Batch
# ============================================================

if run_id is not None:

    incremental_bronze_df = (
        bronze_addresses_df
        .filter(
            (F.col("_load_type") == "INCREMENTAL")
            &
            (F.col("_pipeline_run_id") == run_id)
        )
    )

else:

    incremental_bronze_df = (
        bronze_addresses_df.limit(0)
    )

    print(
        "No incremental address batch "
        "available for processing."
    )


# ============================================================
# 4. Count Current Batch
# ============================================================

incremental_count = (
    incremental_bronze_df.count()
)

print(
    f"Incremental address rows: "
    f"{incremental_count:,}"
)


# ============================================================
# 5. Load Current Silver Addresses
# ============================================================

silver_addresses_df = spark.table(
    "customer360_dev.silver.addresses"
)


# ============================================================
# 6. Clean / Cast Incremental Addresses
# ============================================================

incremental_clean_df = (
    incremental_bronze_df

    .withColumn(
        "address_type_clean",
        F.upper(F.trim("address_type"))
    )

    .withColumn(
        "address_line1_clean",
        F.trim("address_line1")
    )

    .withColumn(
        "address_line2_clean",
        F.trim("address_line2")
    )

    .withColumn(
        "city_clean",
        F.initcap(F.trim("city"))
    )

    .withColumn(
        "state_clean",
        F.initcap(F.trim("state"))
    )

    .withColumn(
        "postal_code_clean",
        F.trim("postal_code")
    )

    .withColumn(
        "country_clean",
        F.upper(F.trim("country"))
    )

    .withColumn(
        "is_primary_cast",
        F.col("is_primary").cast("boolean")
    )

    .withColumn(
        "effective_from_cast",
        F.coalesce(
            F.try_to_timestamp("effective_from"),
            F.to_timestamp("effective_from", "dd-MM-yyyy")
        )
    )

    .withColumn(
        "effective_to_cast",
        F.coalesce(
            F.try_to_timestamp("effective_to"),
            F.to_timestamp("effective_to", "dd-MM-yyyy")
        )
    )

    .withColumn(
        "is_current_cast",
        F.col("is_current").cast("boolean")
    )

    .withColumn(
        "created_at_cast",
        F.coalesce(
            F.try_to_timestamp("created_at"),
            F.to_timestamp("created_at", "dd-MM-yyyy HH:mm")
        )
    )

    .withColumn(
        "updated_at_cast",
        F.coalesce(
            F.try_to_timestamp("updated_at"),
            F.to_timestamp("updated_at", "dd-MM-yyyy HH:mm")
        )
    )
)


# ============================================================
# 7. Deduplicate Current Address Batch
# ============================================================

address_window = (
    Window
    .partitionBy("address_id")
    .orderBy(
        F.col("updated_at_cast").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc()
    )
)

incremental_dedup_df = (
    incremental_clean_df
    .withColumn(
        "_row_number",
        F.row_number().over(address_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)


# ============================================================
# 8. Join Incremental Addresses to Current Silver
# ============================================================

comparison_df = (
    incremental_dedup_df.alias("src")
    .join(
        silver_addresses_df.alias("tgt"),

        F.col("src.address_id")
        ==
        F.col("tgt.address_id"),

        how="left"
    )
)


# ============================================================
# 9. Detect Address Changes
# ============================================================

change_condition = (

    ~F.col("src.customer_id").eqNullSafe(
        F.col("tgt.customer_id")
    )

    |

    ~F.col("src.address_type_clean").eqNullSafe(
        F.col("tgt.address_type")
    )

    |

    ~F.col("src.address_line1_clean").eqNullSafe(
        F.col("tgt.address_line1")
    )

    |

    ~F.col("src.address_line2_clean").eqNullSafe(
        F.col("tgt.address_line2")
    )

    |

    ~F.col("src.city_clean").eqNullSafe(
        F.col("tgt.city")
    )

    |

    ~F.col("src.state_clean").eqNullSafe(
        F.col("tgt.state")
    )

    |

    ~F.col("src.postal_code_clean").eqNullSafe(
        F.col("tgt.postal_code")
    )

    |

    ~F.col("src.country_clean").eqNullSafe(
        F.col("tgt.country")
    )

    |

    ~F.col("src.is_primary_cast").eqNullSafe(
        F.col("tgt.is_primary")
    )

    |

    ~F.col("src.effective_from_cast").eqNullSafe(
        F.col("tgt.effective_from")
    )

    |

    ~F.col("src.effective_to_cast").eqNullSafe(
        F.col("tgt.effective_to")
    )

    |

    ~F.col("src.is_current_cast").eqNullSafe(
        F.col("tgt.is_current")
    )
)


# ============================================================
# 10. Classify INSERT / UPDATE / UNCHANGED
# ============================================================

classified_df = (
    comparison_df
    .withColumn(
        "cdc_action",

        F.when(
            F.col("tgt.address_id").isNull(),
            F.lit("INSERT")
        )

        .when(
            change_condition,
            F.lit("UPDATE")
        )

        .otherwise(
            F.lit("UNCHANGED")
        )
    )
)


# ============================================================
# 11. Build Clean Address CDC Output
# ============================================================

address_cdc_df = (
    classified_df
    .select(
        F.col("src.address_id").alias("address_id"),
        F.col("src.customer_id").alias("customer_id"),
        F.col("src.address_type_clean").alias("address_type"),
        F.col("src.address_line1_clean").alias("address_line1"),
        F.col("src.address_line2_clean").alias("address_line2"),
        F.col("src.city_clean").alias("city"),
        F.col("src.state_clean").alias("state"),
        F.col("src.postal_code_clean").alias("postal_code"),
        F.col("src.country_clean").alias("country"),
        F.col("src.is_primary_cast").alias("is_primary"),
        F.col("src.effective_from_cast").alias("effective_from"),
        F.col("src.effective_to_cast").alias("effective_to"),
        F.col("src.is_current_cast").alias("is_current"),
        F.col("src.created_at_cast").alias("created_at"),
        F.col("src.updated_at_cast").alias("updated_at"),

        F.col("src._source_file").alias("_source_file"),
        F.col("src._source_system").alias("_source_system"),
        F.col("src._load_type").alias("_load_type"),
        F.col("src._ingestion_timestamp").alias(
            "_ingestion_timestamp"
        ),
        F.col("src._pipeline_run_id").alias(
            "_pipeline_run_id"
        ),

        F.col("cdc_action")
    )
)


# ============================================================
# 12. CDC Summary
# ============================================================

if incremental_count > 0:

    (
        address_cdc_df
        .groupBy("cdc_action")
        .count()
        .orderBy("cdc_action")
        .show()
    )

else:

    print(
        "No incremental address rows "
        "available for CDC classification."
    )


# ============================================================
# 13. Preview
# ============================================================

display(
    address_cdc_df.limit(50)
)


# ============================================================
# 14. Persist Address CDC Work Table
# ============================================================

(
    address_cdc_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.control.address_cdc_work"
    )
)

Processing Address Run ID: None
No incremental address batch available for processing.
Incremental address rows: 0
No incremental address rows available for CDC classification.


address_id,customer_id,address_type,address_line1,address_line2,city,state,postal_code,country,is_primary,effective_from,effective_to,is_current,created_at,updated_at,_source_file,_source_system,_load_type,_ingestion_timestamp,_pipeline_run_id,cdc_action
